In [ ]:
!pip install figuard -q
print("✓ FiGuard installed")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient(
    api_key="sb_live_demo",
    base_url="https://sandbox.figuard.io",
)

budget = client.create_budget(
    user_id="agent_001",
    total_limit=100_000,
    unit="tokens",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="LLM research session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   {budget.total_limit:,} tokens")

In [ ]:
# Authorized LLM call
auth = client.authorize(
    session_token=budget.session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="GPT-4o: summarize research paper",
    requested_quantity=8_500,
    idempotency_key="llm-call-001",
)
print(f"LLM call 1: {auth.decision} — {auth.approved_quantity:,} tokens")
client.confirm_event(auth.event_id, confirmed_quantity=8_200)
print("✓ Confirmed. 8,200 tokens consumed.")

# Large call — within budget still
auth2 = client.authorize(
    session_token=budget.session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="Claude 3.5: long-form synthesis",
    requested_quantity=45_000,
    idempotency_key="llm-call-002",
)
print(f"LLM call 2: {auth2.decision} — {auth2.approved_quantity:,} tokens")
client.confirm_event(auth2.event_id, confirmed_quantity=44_100)
print("✓ Confirmed. 44,100 tokens consumed.")

# Over budget
auth3 = client.authorize(
    session_token=budget.session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="o1: full document re-analysis",
    requested_quantity=60_000,
    idempotency_key="llm-call-003",
)
print(f"LLM call 3: {auth3.decision} — {auth3.denial_reason}")
print("Budget enforced. No tokens consumed.")

print(f"\n→ See the spend tree: https://sandbox.figuard.io/ui")